[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/02-data-harmonization/02-alias_sets_nicknames_and_synonyms.ipynb)

# Alias Sets: Nicknames & Synonyms

`CharacterMapping` handles *spelling* variation, accents, casing, umlauts. But some variation isn't about spelling at all. "Bob" and "Robert" don't share most of their letters, yet they refer to the same person. "IBM" and "International Business Machines" are the same company, but one isn't even a partial match for the other character-by-character.

No amount of fuzzy character matching solves this, because the words themselves are genuinely different. What you need is a dictionary of known equivalences, and that's exactly what `AliasSet` provides.

In this notebook you will:

1. See why a nickname query fails against a full-name index with no alias support
2. Create an `AliasSet` and attach it to a field
3. Understand how the `penalty` parameter affects match confidence
4. Apply aliases to a second use case: company name abbreviations
5. Import and export alias sets as JSON, dictionaries, and DataFrames


## Install the SDK

M|BOX is distributed on PyPI. Run the cell below to install it (or run this in your terminal without the `!`).

In [ ]:
# !pip install mbox

## 1. The problem: nicknames aren't typos

Let's index a small set of customers by their legal first name, then search using the nickname someone would actually type.

In [24]:
import os 

import pandas as pd
from mbox.indexing import TableIndexer

df = pd.DataFrame({
    "customer_id": ["C3001", "C3002", "C3003", "C3004"],
    "first_name": ["Robert", "Katharina", "Alexander", "Elizabeth"],
    "last_name": ["Hayes", "Voss", "Kim", "Turner"]
})

df

,customer_id,first_name,last_name
0,C3001,Robert,Hayes
1,C3002,Katharina,Voss
2,C3003,Alexander,Kim
3,C3004,Elizabeth,Turner


Let's build an index from this DataFrame and search for "Bob Hayes." In American English, "Bob" is a common nickname for "Robert", but nothing in the raw text connects the two.

In [25]:
baseline_index = TableIndexer.create_index(df, index_columns=["first_name", "last_name"], tmp_dir = "tmp_index")

baseline_results = baseline_index.match(
    first_name="Bob",
    last_name="Hayes",
    include_field_scores=True
)

baseline_results

,query_row,index_row,first_name_candidate,last_name_candidate,customer_id_candidate,overall_score,first_name_score,last_name_score
0,0,-1,,,,0,0,0


As you can see we could not find any match in our data. "Bob" and "Robert" share exactly one letter in common ("b"), so there's very little for `APPROX` fuzzy matching to work with. This isn't a spelling problem `CharacterMapping` can fix. "Bob" isn't a misspelling of "Robert", it's a *different word* that happens to mean the same person. That equivalence has to be taught to the engine explicitly.

## 2. Teaching the engine an equivalence with `AliasSet`

`AliasSet` stores a list of `(word, alias, penalty)` mappings. `word` is the canonical or target term, `alias` is the alternative someone might actually type, and `penalty` is how much match quality to deduct when a result is found *via* the alias rather than an exact or fuzzy literal match.

Let's create one for common English first-name nicknames.

In [26]:
from mbox.aliases import AliasSet

first_name_aliases = AliasSet(name="given_names")
first_name_aliases.add(word="Robert", alias="Bob", penalty=0)
first_name_aliases.add(word="Katharina", alias="Kathi", penalty=5)
first_name_aliases.add(word="Alexander", alias="Alex", penalty=2)
first_name_aliases.add(word="Elizabeth", alias="Liz", penalty=5)
first_name_aliases.add(word="Elizabeth", alias="Beth", penalty=5)

AliasSet(name='given_names', aliases=OrderedDict({'{"word":"Robert","alias":"Bob"}': 0, '{"word":"Katharina","alias":"Kathi"}': 5, '{"word":"Alexander","alias":"Alex"}': 2, '{"word":"Elizabeth","alias":"Liz"}': 5, '{"word":"Elizabeth","alias":"Beth"}': 5}))

`add()` returns `self`, so you can chain calls if you prefer:

```python
first_name_aliases = (
    AliasSet(name="given_names")
    .add(word="Robert", alias="Bob", penalty=0)
    .add(word="Katharina", alias="Kathi", penalty=5)
)
```

Notice `"Elizabeth"` has *two* aliases here, `"Liz"` and `"Beth"`. An `AliasSet` isn't limited to one nickname per word; real names often have several.

## 3. Attach the alias set and re-run the query

Now rebuild the index with `alias_sets` attached, and run the exact same nickname query from Step 1.

In [27]:
aliased_index = TableIndexer.create_index(
    df=df,
    index_columns=["first_name", "last_name"],
    alias_sets={"first_name": first_name_aliases},
    tmp_dir="tmp_index"
)

aliased_results = aliased_index.match(
    first_name="Bob",
    last_name="Hayes",
    include_field_scores=True
)

aliased_results

,query_row,index_row,first_name_candidate,last_name_candidate,customer_id_candidate,overall_score,first_name_score,last_name_score
0,0,0,Robert,Hayes,C3001,100,100,100


Now as you can see, `"Bob"` resolved directly to `"Robert"` through the alias, rather than being scored on raw character overlap.

Notice we set `penalty=0` for `"Robert" → "Bob"`. That's a deliberate choice: "Bob" is such a standard, universally recognized nickname for "Robert" that treating it as a full-confidence match makes sense. Compare that to `"Kathi"`, where we used `penalty=5`, still a strong match, but a slightly less universal or more informal nickname, so the score is nudged down a little to reflect that.

In [28]:
# Compare a low-penalty nickname against a higher-penalty one, side by side
comparison = aliased_index.match(
    first_name=["Bob", "Kathi"],
    last_name=["Hayes", "Voss"],
    include_field_scores=True,
    include_queries=True
)

comparison

,query_row,first_name_query,last_name_query,index_row,first_name_candidate,last_name_candidate,customer_id_candidate,overall_score,first_name_score,last_name_score
0,0,Bob,Hayes,0,Robert,Hayes,C3001,100,100,100
1,1,Kathi,Voss,1,Katharina,Voss,C3002,97,95,100


Even though both `"Bob"` and `"Kathi"` are valid, recognized aliases, `"Kathi"`'s slightly higher penalty should pull its `first_name_score` a bit lower than `"Bob"`'s. This is the point of the `penalty` parameter: not every alias deserves the same confidence, and you get to encode that judgment directly.

## 4. A second use case: company abbreviations

`AliasSet` isn't limited to personal names. Corporate data is full of the same problem, abbreviations, legal suffixes, and shorthand that share little in common with the full name they stand for.

In [29]:
companies_df = pd.DataFrame({
    "company_id": ["V001", "V002", "V003"],
    "company_name": ["International Business Machines", "Volkswagen Aktiengesellschaft", "General Electric"]
})

company_aliases = AliasSet(name="corporate_synonyms")
company_aliases.add(word="International Business Machines", alias="IBM", penalty=0)
company_aliases.add(word="Volkswagen Aktiengesellschaft", alias="VW", penalty=0)
company_aliases.add(word="Volkswagen Aktiengesellschaft", alias="Volkswagen AG", penalty=0)
company_aliases.add(word="General Electric", alias="GE", penalty=0)

company_index = TableIndexer.create_index(
    df=companies_df,
    index_columns=["company_name"],
    alias_sets={"company_name": company_aliases},
    tmp_dir = "tmp_index"
)

company_index.match(company_name="IBM", include_field_scores=True)

,query_row,index_row,company_name_candidate,company_id_candidate,overall_score,company_name_score
0,0,0,International Business Machines,V001,100,100


`"IBM"` and `"International Business Machines"` don't overlap meaningfully as strings at all, this match is only possible because the alias explicitly connects them. This is the same mechanism as the nickname example, just applied to a different domain.

## 5. Reusing alias sets across projects

Building an `AliasSet` by hand with repeated `.add()` calls works for small examples, but production alias dictionaries are often large, maintained separately, and shared across services. `AliasSet` supports three ways to move data in and out.

### JSON

The most common option for version-controlling an alias dictionary alongside your code.

In [30]:
os.makedirs("aliases", exist_ok=True)

first_name_aliases.to_json(r"./aliases/first_name_aliases.json")

# Reload in a different notebook, service, or pipeline run
loaded_aliases = AliasSet.from_json("./aliases/first_name_aliases.json")
loaded_aliases.to_dict()

{'name': 'given_names',
 'aliases': {'{"word":"Robert","alias":"Bob"}': 0,
  '{"word":"Katharina","alias":"Kathi"}': 5,
  '{"word":"Alexander","alias":"Alex"}': 2,
  '{"word":"Elizabeth","alias":"Liz"}': 5,
  '{"word":"Elizabeth","alias":"Beth"}': 5}}

### Python dictionaries

Useful when aliases are coming from an API response or need to be constructed dynamically at runtime.

In [31]:
alias_dict = first_name_aliases.to_dict()
rebuilt_aliases = AliasSet.from_dict(alias_dict)
rebuilt_aliases.to_dict()

{'name': 'given_names',
 'aliases': {'{"word":"Robert","alias":"Bob"}': 0,
  '{"word":"Katharina","alias":"Kathi"}': 5,
  '{"word":"Alexander","alias":"Alex"}': 2,
  '{"word":"Elizabeth","alias":"Liz"}': 5,
  '{"word":"Elizabeth","alias":"Beth"}': 5}}

### Pandas DataFrames

Handy when you already maintain nicknames or synonyms in a spreadsheet, database table, or CSV export. The DataFrame needs exactly three columns: `word`, `alias`, and `penalty`.

In [32]:
alias_table = pd.DataFrame({
    "word": ["Robert", "Katharina", "Alexander", "Elizabeth", "Elizabeth"],
    "alias": ["Bob", "Kathi", "Alex", "Liz", "Beth"],
    "penalty": [0, 5, 2, 5, 5]
})

df_aliases = AliasSet.from_dataframe(df=alias_table, name="given_names_from_table")

# Convert back to a DataFrame at any time, e.g. for review or editing
df_aliases.to_df()

,word,alias,penalty
0,Robert,Bob,0
1,Katharina,Kathi,5
2,Alexander,Alex,2
3,Elizabeth,Liz,5
4,Elizabeth,Beth,5


## 6. Choosing penalty values

There's no single correct penalty for every alias, it depends on how confident you are that the alias really does mean the same thing. A rough guideline:

| Penalty | Use for |
|---|---|
| `0` | Exact linguistic equivalents, common diminutives, or standard abbreviations, `"Street"` → `"St"`, `"Robert"` → `"Bob"` |
| `5`–`15` | Informal nicknames or terms with slight semantic divergence, `"Katharina"` → `"Kathi"` |
| `20+` | Broader category generalizations or loose associations, where match confidence should drop meaningfully |

When in doubt, start with a low penalty for well-established equivalences and reserve higher penalties for aliases you're less certain about, you can always tune penalties later once you see how they affect real match results.

## Next steps

- **`building_a_custom_character_mapping.ipynb`**, write your own spelling-normalization rules, complementary to what `AliasSet` handles
- **`combining_mappings_and_aliases_in_one_index.ipynb`**, use `CharacterMapping` and `AliasSet` together on the same field
- **`03-index-configuration/`**, attach aliases explicitly via `TableFieldConfig` as part of a versioned schema

*M|BOX is currently in `beta`. Breaking changes may occur in minor releases until version `1.0.0`.*